# Querying the OTP server

Queries the local OpenTripPlanner GraphQL API (`http://localhost:8080/otp/gtfs/v1`).
Start the server first with `PowerShell Scripts/run_otp.ps1`.

In [1]:
import requests
import pandas as pd

OTP_URL = "http://localhost:8080/otp/gtfs/v1"

def query_otp(query: str, variables: dict | None = None) -> dict:
    """Run a GraphQL query against the OTP server and return the `data` payload."""
    resp = requests.post(
        OTP_URL,
        json={"query": query, "variables": variables or {}},
        headers={"Content-Type": "application/json"},
        timeout=60,
    )
    resp.raise_for_status()
    payload = resp.json()
    if "errors" in payload:
        raise RuntimeError(payload["errors"])
    return payload["data"]

## Example: plan a trip (Melbourne -> Sydney) with fares

In [2]:
PLAN_QUERY = """
query Plan($from: InputCoordinates!, $to: InputCoordinates!, $date: String!, $time: String!) {
  plan(
    from: $from
    to: $to
    date: $date
    time: $time
    transportModes: [{ mode: TRANSIT }, { mode: WALK }]
  ) {
    itineraries {
      duration
      legs {
        mode
        startTime
        endTime
        route { shortName longName }
        from { name }
        to { name }
        fareProducts {
          product {
            name
            ... on DefaultFareProduct {
              price { amount currency { code } }
            }
          }
        }
      }
    }
  }
}
"""

variables = {
    "from": {"lat": -37.81, "lon": 144.96},  # Melbourne
    "to": {"lat": -33.87, "lon": 151.21},    # Sydney
    "date": "2026-05-20",
    "time": "08:00",
}

data = query_otp(PLAN_QUERY, variables)
itineraries = data["plan"]["itineraries"]
print(f"{len(itineraries)} itineraries found")

2 itineraries found


In [3]:
# Flatten legs across all itineraries into a DataFrame
rows = []
for i, itin in enumerate(itineraries):
    for leg in itin["legs"]:
        fare = None
        if leg["fareProducts"]:
            price = leg["fareProducts"][0]["product"].get("price")
            if price:
                fare = f"{price['amount']} {price['currency']['code']}"
        rows.append({
            "itinerary": i,
            "mode": leg["mode"],
            "route": leg["route"]["shortName"] if leg["route"] else None,
            "from": leg["from"]["name"],
            "to": leg["to"]["name"],
            "start": leg["startTime"],
            "end": leg["endTime"],
            "fare": fare,
        })

df = pd.DataFrame(rows)
df

,itinerary,mode,route,from,to,start,end,fare
0,0,WALK,NaN,Origin,Melbourne Central Station,1779228591000,1779228960000,None
1,0,RAIL,Frankston,Melbourne Central Station,Southern Cross Station,1779228960000,1779229200000,None
2,0,WALK,NaN,Southern Cross Station,"Southern Cross Station, Platform 1",1779229200000,1779229587000,None
3,0,RAIL,624,"Southern Cross Station, Platform 1","Central Station, Platform 3",1779229800000,1779270480000,None
4,0,WALK,NaN,"Central Station, Platform 3","Central Station, Platform 21",1779270480000,1779270801000,None
5,0,RAIL,T2,"Central Station, Platform 21","St James Station, Platform 1",1779270961000,1779271230000,None
6,0,WALK,NaN,"St James Station, Platform 1",Destination,1779271230000,1779271553000,None
7,1,WALK,NaN,Origin,Melbourne Central Station,1779228591000,1779228960000,None
8,1,RAIL,Frankston,Melbourne Central Station,Southern Cross Station,1779228960000,1779229200000,None
9,1,WALK,NaN,Southern Cross Station,"Southern Cross Station, Platform 1",1779229200000,1779229587000,None
